In [1]:
# ============================================================
# CUSTOMER CHURN INTELLIGENCE
# 05 - MODEL DEPLOYMENT PREPARATION
# ============================================================

# %% [markdown]
# # Customer Churn Intelligence
# ## 05. Model Deployment Preparation
#
# Objective:
# - Train the production model
# - Save the preprocessing objects
# - Save the trained model
# - Create a reusable prediction function
# - Generate customer risk predictions

# %%
import pandas as pd
import numpy as np
import joblib
import os

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

print("Libraries imported successfully.")

# %% [markdown]
# ## 1. Load Processed Dataset

# %%
df = pd.read_csv(
    "../data/processed/telco_customer_churn_clean.csv"
)

print("Dataset shape:", df.shape)

# %% [markdown]
# ## 2. Prepare Features
#
# We recreate the same feature engineering used during training.

# %%
model_df = df.copy()

# Remove identifier
model_df = model_df.drop(
    columns=["customerID"]
)

# %%
# Tenure features

model_df["TenureYears"] = (
    model_df["tenure"] / 12
)

model_df["IsNewCustomer"] = (
    model_df["tenure"] <= 6
).astype(int)

model_df["IsLongTermCustomer"] = (
    model_df["tenure"] >= 48
).astype(int)

# %%
# Revenue features

model_df["EstimatedAnnualCharges"] = (
    model_df["MonthlyCharges"] * 12
)

model_df["ChargePerTenureMonth"] = (
    model_df["TotalCharges"] /
    model_df["tenure"].replace(0, np.nan)
)

model_df["ChargePerTenureMonth"] = (
    model_df["ChargePerTenureMonth"]
    .fillna(model_df["MonthlyCharges"])
)

# %%
# Service features

service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

for column in service_columns:

    model_df[column + "_Flag"] = (
        model_df[column] == "Yes"
    ).astype(int)

model_df["TotalAdditionalServices"] = (
    model_df[
        [column + "_Flag" for column in service_columns]
    ].sum(axis=1)
)

# %%
# Family feature

model_df["HasFamily"] = (
    (model_df["Partner"] == 1) |
    (model_df["Dependents"] == 1)
).astype(int)

# %%
# Monthly charge feature

monthly_charge_median = (
    model_df["MonthlyCharges"].median()
)

model_df["HighMonthlyCharge"] = (
    model_df["MonthlyCharges"] >
    monthly_charge_median
).astype(int)

# %%
# Contract feature

model_df["MonthToMonth"] = (
    model_df["Contract"] ==
    "Month-to-month"
).astype(int)

# %%
# Payment feature

model_df["ElectronicCheck"] = (
    model_df["PaymentMethod"] ==
    "Electronic check"
).astype(int)

# %% [markdown]
# ## 3. Separate Features and Target

# %%
X = model_df.drop(
    columns=["Churn"]
)

y = model_df["Churn"]

# %% [markdown]
# ## 4. One-Hot Encode Categorical Features

# %%
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

X_encoded = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Final feature count:", X_encoded.shape[1])

# %% [markdown]
# ## 5. Train/Test Split

# %%
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# %% [markdown]
# ## 6. Scale Numerical Features

# %%
scaler = StandardScaler()

scale_columns = [
    column
    for column in [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "TenureYears",
        "EstimatedAnnualCharges",
        "ChargePerTenureMonth"
    ]
    if column in X_train.columns
]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_columns] = (
    scaler.fit_transform(
        X_train[scale_columns]
    )
)

X_test_scaled[scale_columns] = (
    scaler.transform(
        X_test[scale_columns]
    )
)

# %% [markdown]
# ## 7. Train Production Model
#
# Random Forest is used here as the production model.
# We can change this later based on our model evaluation.

# %%
production_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_split=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

production_model.fit(
    X_train_scaled,
    y_train
)

print("Production model trained successfully.")

# %% [markdown]
# ## 8. Create Models Directory

# %%
os.makedirs(
    "../models",
    exist_ok=True
)

# %% [markdown]
# ## 9. Save Model

# %%
joblib.dump(
    production_model,
    "../models/churn_model.pkl"
)

print("Model saved.")

# %% [markdown]
# ## 10. Save Scaler

# %%
joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

print("Scaler saved.")

# %% [markdown]
# ## 11. Save Feature Columns

# %%
feature_columns = X_train_scaled.columns.tolist()

joblib.dump(
    feature_columns,
    "../models/feature_columns.pkl"
)

print(
    f"Saved {len(feature_columns)} feature columns."
)

# %% [markdown]
# ## 12. Save Configuration

# %%
model_config = {
    "monthly_charge_median": monthly_charge_median,
    "scale_columns": scale_columns,
    "service_columns": service_columns
}

joblib.dump(
    model_config,
    "../models/model_config.pkl"
)

print("Model configuration saved.")

# %% [markdown]
# ## 13. Test Saved Model

# %%
loaded_model = joblib.load(
    "../models/churn_model.pkl"
)

loaded_scaler = joblib.load(
    "../models/scaler.pkl"
)

loaded_features = joblib.load(
    "../models/feature_columns.pkl"
)

print("Saved model loaded successfully.")

# %% [markdown]
# ## 14. Generate Test Predictions

# %%
test_probabilities = loaded_model.predict_proba(
    X_test_scaled
)[:, 1]

test_predictions = (
    test_probabilities >= 0.50
).astype(int)

# %%
prediction_summary = pd.DataFrame({
    "Actual": y_test.values,
    "Prediction": test_predictions,
    "ChurnProbability": test_probabilities
})

display(
    prediction_summary.head(20)
)

# %% [markdown]
# ## 15. Create Risk Categories

# %%
prediction_summary["RiskLevel"] = pd.cut(
    prediction_summary["ChurnProbability"],
    bins=[0, 0.30, 0.60, 1.0],
    labels=[
        "Low",
        "Medium",
        "High"
    ],
    include_lowest=True
)

display(
    prediction_summary.head(20)
)

# %% [markdown]
# ## 16. Risk Distribution

# %%
display(
    prediction_summary["RiskLevel"]
    .value_counts()
)

# %% [markdown]
# ## 17. Deployment Files

# %%
print("Deployment files:")

for file in os.listdir("../models"):
    print("-", file)

# %% [markdown]
# # Deployment Preparation Completed
#
# Saved:
#
# - churn_model.pkl
# - scaler.pkl
# - feature_columns.pkl
# - model_config.pkl
#
# These files will be used by the Streamlit application.

Libraries imported successfully.
Dataset shape: (7043, 21)
Final feature count: 46


C:\Users\acer\AppData\Local\Temp\ipykernel_2372\3716761303.py:158: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


Production model trained successfully.
Model saved.
Scaler saved.
Saved 46 feature columns.
Model configuration saved.
Saved model loaded successfully.


,Actual,Prediction,ChurnProbability
0,0,0,0.008456
1,0,1,0.859537
2,0,0,0.163880
3,0,1,0.588445
4,0,0,0.016115
5,0,1,0.722468
6,0,1,0.568347
7,0,0,0.163321
8,0,0,0.047917
9,1,1,0.673696


,Actual,Prediction,ChurnProbability,RiskLevel
0,0,0,0.008456,Low
1,0,1,0.859537,High
2,0,0,0.163880,Low
3,0,1,0.588445,Medium
4,0,0,0.016115,Low
5,0,1,0.722468,High
6,0,1,0.568347,Medium
7,0,0,0.163321,Low
8,0,0,0.047917,Low
9,1,1,0.673696,High


RiskLevel
Low       660
High      423
Medium    326
Name: count, dtype: int64

Deployment files:
- churn_model.pkl
- feature_columns.pkl
- model_config.pkl
- scaler.pkl
